# 04 — The four objects: data, target, model, loss

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/004-supervised-learning.ipynb)

Every learning problem is built from four things: a table of inputs, a column of targets, a model that turns inputs into a prediction, and a loss that scores how wrong the prediction is. Here you build all four by hand on ten rows, watch the loss fall as the weight moves, and see why we usually average the loss instead of summing it.

## 1. The four objects, printed

Ten rows, two features each. The target was built from a straight-line rule plus a little noise, so a correct model should recover that rule and leave only the noise behind. The model is the dot product plus a bias, and the loss is the squared gap between prediction and target.

In [ ]:
import numpy as np                                       # the array library
x1 = np.arange(1, 11) * 0.5                              # feature 1: 0.5, 1.0, ... 5.0
x2 = np.array([0.5, -0.2, 1.0, 0.3, -0.8, 1.2, 0.1, -0.4, 0.9, -1.1])   # feature 2, ten small numbers
X = np.column_stack([x1, x2])                            # the design matrix: 10 rows, 2 columns
noise = np.array([0.1, -0.2, 0.05, 0.3, -0.1, 0.2, -0.05, 0.15, -0.25, 0.1])   # measurement noise
y = 2.0 * x1 - 1.0 * x2 + 0.5 + noise                    # the target: a straight rule plus noise
w = np.array([2.0, -1.0])                                # the two weights we will test
b = 0.5                                                  # the bias: one constant, shared by all rows
print("X:", X.shape, " y:", y.shape, " w:", w)           # shapes keep us honest
print("row 0 of X:", X[0], "-> y[0] =", round(y[0], 3))  # one row and the target it should hit
f = X @ w + b                                            # the model, applied to all ten rows at once
print("first three f:", np.round(f[:3], 3))              # predictions
print("first three y:", np.round(y[:3], 3))              # the values they are chasing

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `x1 = ...` — the first feature, ten evenly spaced values.
- `x2 = ...` — the second feature, ten small numbers of either sign.
- `X = np.column_stack(...)` — glues the two feature columns into one 10-by-2 table.
- `noise = ...` — one small error per row, fixed so the notebook is reproducible.
- `y = ...` — the target: a straight rule in the two features plus that noise.
- `w = ...` — the two weights of the model, one per feature.
- `b = 0.5` — the bias, a single constant added to every prediction.
- `print("X:", ...)` — confirms ten rows, two columns, two weights.
- `print("row 0 ...")` — one concrete row and its target.
- `f = X @ w + b` — the model: a dot product per row, then the bias.
- `print("first three f ...")` — the predictions.
- `print("first three y ...")` — the targets they are chasing.

## 2. Step by step: the loss, one row at a time

The loss asks how far each prediction sits from its target, squares that gap, and adds the squares up. Doing it row by row shows that the objective is just one number summarising ten mistakes — the smaller it is, the better the weights fit.

In [ ]:
total = 0.0                                             # the running sum of squared errors
for i in range(len(y)):                                  # one iteration per row
    pred = X[i] @ w + b                                  # predict row i: a dot product plus the bias
    err = pred - y[i]                                    # residual: prediction minus target
    sq = err ** 2                                        # squared loss for this one row
    total = total + sq                                   # fold it into the running total
    print(f"row {i}: pred={pred:.3f} err={err:+.3f} sum={total:.3f}")   # watch the sum grow
print("sum of squares:", round(total, 4))                # the objective on the sum scale
print("average:", round(total / len(y), 4))              # the objective on the per-row scale

**Line by line**

- `total = 0.0` — the accumulator that becomes the objective.
- `for i in range(len(y))` — one pass over the ten rows.
- `pred = X[i] @ w + b` — the model's prediction for a single row.
- `err = pred - y[i]` — the residual, signed so we can see over- and under-shoots.
- `sq = err ** 2` — squaring removes the sign and punishes big misses hardest.
- `total = total + sq` — the loss is additive across rows.
- `print(f"row {i} ...")` — the running total after each row.
- `print("sum of squares"...)` — the final objective, sum version.
- `print("average"...)` — the same number divided by ten rows.

## 3. Plot: the loss is a bowl

Hold the second weight and the bias fixed and sweep the first weight. Squared error is a quadratic in the weight, so the curve is a parabola: one lowest point, and the slope tells you which way to move. The dashed line marks the weight that generated the data.

In [ ]:
import matplotlib.pyplot as plt                          # plotting library
grid = np.linspace(-1, 5, 61)                            # candidate values for the first weight
losses = []                                              # one loss per candidate
for g in grid:                                           # walk the candidates in order
    w_try = np.array([g, w[1]])                          # vary weight 1, hold weight 2 fixed
    resid = X @ w_try + b - y                            # residuals for all ten rows at once
    losses.append(np.mean(resid ** 2))                   # average squared loss at this weight
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.plot(grid, losses, color="tab:blue")                 # the bowl
plt.axvline(2.0, color="grey", linestyle="--")           # the weight that built the target
plt.xlabel("weight on feature 1")                        # x-axis label
plt.ylabel("average squared loss")                       # y-axis label
plt.title("One lowest point: the best weight is where the bowl bottoms out")   # the takeaway
plt.show()                                               # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — Colab ships matplotlib.
- `grid = np.linspace(-1, 5, 61)` — 61 trial values of the first weight.
- `losses = []` — empty list to collect one number per trial.
- `for g in grid` — one iteration per trial weight.
- `w_try = ...` — a weight vector with the first entry replaced by `g`.
- `resid = ...` — all ten residuals, computed without a loop.
- `losses.append(...)` — the mean squared residual at this weight.
- `plt.figure(...)` — a wide, short figure.
- `plt.plot(...)` — loss against weight; the shape is a parabola.
- `plt.axvline(2.0, ...)` — marks the true weight for comparison.
- `plt.xlabel(...)` — the quantity being varied.
- `plt.ylabel(...)` — the quantity being measured.
- `plt.title(...)` — the one sentence to remember.
- `plt.show()` — renders the figure.

## 4. Interactive: slide the weight down the bowl

Drag the slider to choose the first weight. The red dot is where your choice sits on the loss curve, and the title reports the loss it earns. Stop when the dot reaches the bottom and check that you land on the true weight.

In [ ]:
from ipywidgets import interact                            # sliders that rerun a function
grid = np.linspace(-1, 5, 61)                            # the same grid as the plot above
curve = np.array([np.mean((X @ np.array([g, w[1]]) + b - y) ** 2) for g in grid])   # loss at every grid point

def show(w1=2.0):                                        # w1 comes from the slider
    loss_now = np.mean((X @ np.array([w1, w[1]]) + b - y) ** 2)   # loss at the chosen weight
    plt.figure(figsize=(6, 3))                           # a fresh figure for each drag
    plt.plot(grid, curve, color="tab:blue")              # the whole bowl
    plt.scatter([w1], [loss_now], color="tab:red", zorder=3)   # you, standing on the bowl
    plt.xlabel("weight on feature 1")                    # x-axis label
    plt.ylabel("average squared loss")                   # y-axis label
    plt.title(f"w1 = {w1:.2f} -> loss = {loss_now:.3f}")  # live readout
    plt.show()                                           # draw the frame

interact(show, w1=(-1.0, 5.0, 0.1))                      # attach the slider: range and step

**Line by line**

- `from ipywidgets import interact` — the widget helper Colab preinstalls.
- `grid = np.linspace(...)` — the same trial weights as section 3.
- `curve = ...` — the loss at all 61 grid points, computed once outside the redraw.
- `def show(w1=2.0)` — the function that redraws one frame per slider value.
- `loss_now = ...` — the loss at the weight you picked.
- `plt.figure(...)` — start a clean figure so frames do not stack.
- `plt.plot(...)` — the fixed bowl, the context for your dot.
- `plt.scatter(...)` — your current weight as a red dot.
- `plt.xlabel(...)` — the weight you are choosing.
- `plt.ylabel(...)` — the loss it earns.
- `plt.title(f"...")` — the live readout of weight and loss.
- `plt.show()` — draws the frame.
- `interact(show, w1=(...))` — slider from -1 to 5 in steps of 0.1.

## 5. When it breaks: sum versus average

Copy the same ten rows ten times and nothing about the problem changes — yet the summed loss multiplies by ten and the averaged loss does not move. Any reported loss must say how many rows it covers, or a bigger dataset will look like a worse model.

In [ ]:
big = np.vstack([X] * 10)                               # the same ten rows, copied ten times
y_big = np.tile(y, 10)                                   # their targets, so 100 rows in total
print("sum   on 10 rows:", round(np.sum((X @ w + b - y) ** 2), 3))          # small number
print("sum  on 100 rows:", round(np.sum((big @ w + b - y_big) ** 2), 3))    # ten times bigger, same data
print("mean on  10 rows:", round(np.mean((X @ w + b - y) ** 2), 3))         # per-row loss
print("mean on 100 rows:", round(np.mean((big @ w + b - y_big) ** 2), 3))   # unchanged, as it should be
positions = np.arange(4)                                 # four bar positions
sums = [np.sum((X @ w + b - y) ** 2), np.sum((big @ w + b - y_big) ** 2)]     # the two sum losses
means = [np.mean((X @ w + b - y) ** 2), np.mean((big @ w + b - y_big) ** 2)]  # the two mean losses
heights = sums + means                                   # bar heights, in the order sum, sum, mean, mean
plt.figure(figsize=(6, 3))                               # a wide, short figure
plt.bar(positions, heights, color=["tab:orange", "tab:orange", "tab:green", "tab:green"])   # sums left, means right
plt.xticks(positions, ["sum n=10", "sum n=100", "mean n=10", "mean n=100"])   # label each bar
plt.ylabel("loss value")                                 # y-axis label
plt.title("Copying rows inflates the sum; the average is stable")   # the takeaway
plt.show()                                               # render the figure

**Line by line**

- `big = np.vstack(...)` — stacks ten copies of the table into 100 rows.
- `y_big = np.tile(...)` — the matching 100 targets.
- `print("sum on 10 rows"...)` — the sum over ten rows.
- `print("sum on 100 rows"...)` — the same data, ten times the sum.
- `print("mean on 10 rows"...)` — the average over ten rows.
- `print("mean on 100 rows"...)` — the same average, to more decimal places.
- `positions = np.arange(4)` — x-coordinates for four bars.
- `sums = [...]` — the two summed losses, ready for the chart.
- `means = [...]` — the two averaged losses.
- `heights = sums + means` — glues the four values into bar order.
- `plt.figure(...)` — a wide, short figure.
- `plt.bar(..., color=[...])` — orange bars are sums, green bars are means.
- `plt.xticks(...)` — write what each bar is underneath it.
- `plt.ylabel(...)` — the units of every bar.
- `plt.title(...)` — the lesson in one line.
- `plt.show()` — renders the figure.

## Your turn

Set `b = 0.0` in section 1 and predict whether section 2's average loss rises or falls before you rerun. Then change `x2[0]` to `4.0` and see how far the bottom of the bowl in section 4 moves.